In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv('/content/EVSE-B-PowerCombined.csv')

# Drop irrelevant columns if needed
df = df[['time', 'shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State', 'Attack', 'Label']]

# Convert labels to binary (0 for normal, 1 for attack)
df['Label'] = df['Label'].apply(lambda x: 1 if x == 'attack' else 0)

# Standardize the features
scaler = StandardScaler()
features = scaler.fit_transform(df[['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW']])
labels = df['Label'].values

# Train-validation-test split
X_train, X_temp, y_train, y_temp = train_test_split(features, labels, test_size=0.4, stratify=labels, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Reshape data for RNN (sequence length = 1 in this case)
X_train_tensor = torch.tensor(X_train[:, None, :], dtype=torch.float32)  # Add a sequence dimension
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val[:, None, :], dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test[:, None, :], dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

# Define the RNN model
class EVChargingRNN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super(EVChargingRNN, self).__init__()
        self.rnn = torch.nn.RNN(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, output_dim)
        self.dropout = torch.nn.Dropout(0.3)

    def forward(self, x):
        # RNN output
        out, _ = self.rnn(x)
        # Use the last hidden state for classification
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return torch.sigmoid(out).squeeze()

# Instantiate the model
model = EVChargingRNN(input_dim=features.shape[1], hidden_dim=16, output_dim=1, num_layers=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.BCELoss()

# Training function
def train():
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch).squeeze()
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

# Evaluation function
def evaluate(loader):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            outputs = model(X_batch).squeeze()
            preds = (outputs > 0.5).float()
            all_preds.append(preds.cpu())
            all_labels.append(y_batch.cpu())
            all_probs.append(outputs.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    roc_auc = None
    if len(set(all_labels)) > 1:
        roc_auc = roc_auc_score(all_labels, all_probs)

    return accuracy, precision, f1, roc_auc

# Training loop with validation
for epoch in range(50):
    loss = train()
    val_accuracy, val_precision, val_f1, val_auc = evaluate(val_loader)
    print(f'Epoch {epoch+1}, Loss: {loss:.4f}, Val Accuracy: {val_accuracy:.4f}, Val Precision: {val_precision:.4f}, Val F1: {val_f1:.4f}, Val AUC: {val_auc:.4f}')

# Test evaluation
test_accuracy, test_precision, test_f1, test_roc_auc = evaluate(test_loader)
print(f'Test Accuracy: {test_accuracy:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test F1 Score: {test_f1:.4f}')
print(f'Test ROC-AUC: {test_roc_auc:.4f}')


<ipython-input-2-533920f2d020>:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Label'] = df['Label'].apply(lambda x: 1 if x == 'attack' else 0)


Epoch 1, Loss: 0.3576, Val Accuracy: 0.8755, Val Precision: 0.8755, Val F1: 0.9336, Val AUC: 0.6895
Epoch 2, Loss: 0.3484, Val Accuracy: 0.8761, Val Precision: 0.8764, Val F1: 0.9339, Val AUC: 0.7139
Epoch 3, Loss: 0.3417, Val Accuracy: 0.8760, Val Precision: 0.8760, Val F1: 0.9338, Val AUC: 0.7357
Epoch 4, Loss: 0.3407, Val Accuracy: 0.8762, Val Precision: 0.8761, Val F1: 0.9340, Val AUC: 0.7418
Epoch 5, Loss: 0.3381, Val Accuracy: 0.8760, Val Precision: 0.8764, Val F1: 0.9338, Val AUC: 0.7419
Epoch 6, Loss: 0.3385, Val Accuracy: 0.8762, Val Precision: 0.8769, Val F1: 0.9339, Val AUC: 0.7451
Epoch 7, Loss: 0.3350, Val Accuracy: 0.8755, Val Precision: 0.8776, Val F1: 0.9334, Val AUC: 0.7690
Epoch 8, Loss: 0.3256, Val Accuracy: 0.8764, Val Precision: 0.8782, Val F1: 0.9339, Val AUC: 0.8069
Epoch 9, Loss: 0.3210, Val Accuracy: 0.8762, Val Precision: 0.8766, Val F1: 0.9339, Val AUC: 0.8092
Epoch 10, Loss: 0.3194, Val Accuracy: 0.8745, Val Precision: 0.8795, Val F1: 0.9327, Val AUC: 0.8407